## Word Level LSTM

### Imports

In [1]:
from keras.models import Sequential
from keras.layers import Activation,LSTM,Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CosineSimilarity
from keras.layers.embeddings import Embedding
from tensorflow.keras.metrics import CosineSimilarity
import pandas as pd
import numpy as np
import re
from gensim.models import Word2Vec

### Helper functions

In [2]:
def split_str(delimiters, string, maxsplit=0):
    regex_pattern = '|'.join(map(re.escape, delimiters))
    return re.split(regex_pattern, string, maxsplit)

### Read data

In [3]:
pop_df = pd.read_csv("./../dataset/preprocessed_pop.csv")
pop_df = pop_df.drop('Unnamed: 0', axis=1)
pop_df = pop_df.drop('Unnamed: 0.1', axis=1)

In [4]:
pop_df.shape

(8370, 5)

In [5]:
# pop_df.loc[0]["lyrics"]
pop_df.head(5)

,artist,genre,title,lyrics,word_num
0,Justin Timberlake,pop,Mirrors,Aren't you somethin' to admire? 'Cause your sh...,968
1,Justin Timberlake,pop,Suit & Tie,"Ooh-oh I be on my suit and tie, shit tied, shi...",723
2,Justin Timberlake,pop,Say Something,"Mhmm, yeah, alright Ooh Everyone knows all ab...",452
3,Justin Timberlake,pop,Pusher Love Girl,Hey little mama Ain't gotta ask me if I want t...,746
4,Justin Timberlake,pop,True Colors,"You with the sad eyes Don't be discouraged, oh...",229


In [6]:
train_sz = 1500
train_pop = pop_df.iloc[:train_sz]
print(train_pop.shape[0])

1500


### Training corpus

In [16]:
words_corpus=''
for index,row in train_pop.iterrows():
    words_corpus += " "
    words_corpus += row["title"]
    lyric = split_str(" ;,-\\\"()[].?!:{}",  row["lyrics"].lower().strip())
    # lyric = row["lyrics"].lower().split(" ")
    lyric = [s for s in lyric if s!=''][0:100]
    words_corpus += " " + " ".join(lyric)
cleaned_corpus = words_corpus.split(" ")
cleaned_corpus.extend([";", ",", "-", "\\", "'", '"', "(", ")", "[", "]",  ".", "?", "!", ":", "{", "}"])


In [18]:
vocab = list(set(cleaned_corpus))
print("Length of vocabulary:",len(vocab))
word_ix={c:i for i,c in enumerate(vocab)}
ix_word={i:c for i,c in enumerate(vocab)}

Length of vocabulary: 9215


In [9]:
# word_ix
# ix_word

In [20]:
import json
with open("./../dataset/word_ix_pop.json", "w") as write_file:
    json.dump(word_ix, write_file, indent=4)
with open("./../dataset/ix_word_pop.json", "w") as write_file:
    json.dump(ix_word, write_file, indent=4)

### Word2vec embeddings

In [26]:
# create corpus of list of sentences 
pop_sentences = []
for index, row in train_pop.iterrows():
    lyric = split_str(" ;,-\\\"()[].?!:{}",  row["lyrics"].lower().strip())
    lyric = [s for s in lyric if s!=''][0:100]
    pop_sentences.append(lyric)

In [27]:
print(len(pop_sentences))
print(pop_sentences[0])

1500
["aren't", 'you', "somethin'", 'to', 'admire', "'cause", 'your', 'shine', 'is', "somethin'", 'like', 'a', 'mirror', 'and', 'i', "can't", 'help', 'but', 'notice', 'you', 'reflect', 'in', 'this', 'heart', 'of', 'mine', 'if', 'you', 'ever', 'feel', 'alone', 'and', 'the', 'glare', 'makes', 'me', 'hard', 'to', 'find', 'just', 'know', 'that', "i'm", 'always', 'parallel', 'on', 'the', 'other', 'side', "'cause", 'with', 'your', 'hand', 'in', 'my', 'hand', 'and', 'a', 'pocket', 'full', 'of', 'soul', 'i', 'can', 'tell', 'you', "there's", 'no', 'place', 'we', "couldn't", 'go', 'just', 'put', 'your', 'hand', 'on', 'the', 'past', "i'm", 'here', 'tryna', 'pull', 'you', 'through', 'you', 'just', 'gotta', 'be', 'strong', 'i', "don't", 'wanna', 'lose', 'you', 'now', "i'm", "lookin'", 'right', 'at']


In [28]:
# save word embedding model
word_model = Word2Vec(sentences=pop_sentences, vector_size=300, window=5, min_count=1, workers=4)
# word_model.save("./../dataset/word2vec_300_pop.model")

In [24]:
word_model = Word2Vec.load("./../dataset/word2vec_300_pop.model")

In [51]:
embed = word_model.wv['a']
print(embed.shape)

(300,)


### Training data

In [29]:
vec_size = 300
num_words = 10

In [32]:
sentences=[]
next_word=[]
for index, row in train_pop.iterrows():
    lyric = split_str(" ;,-\\\"()[].?!:{}",  row["lyrics"].lower().strip())
    lyric = [s for s in lyric if s!=''][0:100]

    for i in range(len(lyric)-num_words-1):        
        sentences.append(" ".join(lyric[i:i+num_words]))
        next_word.append(lyric[i+num_words])

In [33]:
print(len(sentences))
sentences[0:10]

131707


["aren't you somethin' to admire 'cause your shine is somethin'",
 "you somethin' to admire 'cause your shine is somethin' like",
 "somethin' to admire 'cause your shine is somethin' like a",
 "to admire 'cause your shine is somethin' like a mirror",
 "admire 'cause your shine is somethin' like a mirror and",
 "'cause your shine is somethin' like a mirror and i",
 "your shine is somethin' like a mirror and i can't",
 "shine is somethin' like a mirror and i can't help",
 "is somethin' like a mirror and i can't help but",
 "somethin' like a mirror and i can't help but notice"]

In [34]:
print(len(next_word))
print(next_word[0:10])

131707
['like', 'a', 'mirror', 'and', 'i', "can't", 'help', 'but', 'notice', 'you']


### Training data Embeddings

In [35]:
### y as word embed, of vec_size
X=np.zeros((len(sentences),num_words,vec_size))
y=np.zeros((len(sentences),vec_size))
for ix in range(len(sentences)):
    
    y[ix] = word_model.wv[next_word[ix]]
    # y[ix,word_ix[next_word[ix]]] = 1

    words = sentences[ix].split(" ")
    if ix%50000==0:
        print(ix, len(words))
        print(sentences[ix])
        print(words)
    for iy in range(len(words)):
        X[ix,iy] = word_model.wv[words[iy]]

0 10
aren't you somethin' to admire 'cause your shine is somethin'
["aren't", 'you', "somethin'", 'to', 'admire', "'cause", 'your', 'shine', 'is', "somethin'"]
50000 10
girl you fucking with a winner i'm the man i'm
['girl', 'you', 'fucking', 'with', 'a', 'winner', "i'm", 'the', 'man', "i'm"]
100000 10
something 'bout the way that you workin' me teasin' me
['something', "'bout", 'the', 'way', 'that', 'you', "workin'", 'me', "teasin'", 'me']


In [36]:
# y as one-hot vector of vocab_size
X=np.zeros((len(sentences),num_words,vec_size))
y=np.zeros((len(sentences),len(vocab)))
for ix in range(len(sentences)):
    
    # y[ix, ] = word_model.wv[next_word[ix]]
    y[ix,word_ix[next_word[ix]]] = 1

    words = sentences[ix].split(" ")
    if ix%50000==0:
        print(ix, len(words))
        print(sentences[ix])
        print(words)
    for iy in range(len(words)):
        X[ix,iy] = word_model.wv[words[iy]]

0 10
aren't you somethin' to admire 'cause your shine is somethin'
["aren't", 'you', "somethin'", 'to', 'admire', "'cause", 'your', 'shine', 'is', "somethin'"]
50000 10
girl you fucking with a winner i'm the man i'm
['girl', 'you', 'fucking', 'with', 'a', 'winner', "i'm", 'the', 'man', "i'm"]
100000 10
something 'bout the way that you workin' me teasin' me
['something', "'bout", 'the', 'way', 'that', 'you', "workin'", 'me', "teasin'", 'me']


In [37]:
print(X.shape, y.shape)

(131707, 10, 300) (131707, 9215)


### Model

In [41]:
# cosine_loss = CosineSimilarity(axis=1)
# model_emb=Sequential()
# # model_emb.add(Embedding(vec_size, 100, input_length=num_words))
# # model_emb.add(Flatten())
# model_emb.add(LSTM(256, input_shape = (num_words, vec_size)))
# model_emb.add(Dense(300,activation='softmax'))
# embedding = Embedding(input_dim = 1, output_dim = 1, name='embedding')
# model_emb.add(embedding)
# # model_emb.add(Dense(vec_size))
# # model_emb.add(Activation('softmax'))
# model_emb.summary()
# model_emb.compile(optimizer=Adam(learning_rate=0.0001),loss=cosine_loss, metrics = ['accuracy'])

In [42]:
# training
# model_emb.fit(X,y,epochs=5,batch_size=256)

In [38]:
# cosine_loss = CosineSimilarity(axis=1)
model=Sequential()
model.add(LSTM(256,input_shape=(num_words,vec_size)))
model.add(Dense(len(vocab)))
model.add(Activation('softmax'))
model.summary()
model.compile(optimizer=Adam(learning_rate=0.0001),loss='categorical_crossentropy')

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 256)               570368    
                                                                 
 dense (Dense)               (None, 9215)              2368255   
                                                                 
 activation (Activation)     (None, 9215)              0         
                                                                 
Total params: 2,938,623
Trainable params: 2,938,623
Non-trainable params: 0
_________________________________________________________________


In [39]:
# training
model.fit(X,y,epochs=3,batch_size=128)

Epoch 1/3
1029/1029 [==============================] - 152s 139ms/step - loss: 6.4600
Epoch 2/3
1029/1029 [==============================] - 173s 168ms/step - loss: 6.1198
Epoch 3/3
1029/1029 [==============================] - 215s 208ms/step - loss: 6.0129


In [60]:
#serialize model to JSON  serialize model to JSON
# model_json = model.to_json()
# with open("model.json", "w") as json_file:
#     json_file.write(model_json)
# serialize weights to HDF5
model.save_weights("./../dataset/models/model_word_300_pop.h5")
print("Saved model to disk")

Saved model to disk


### Inference

In [41]:
# import random
np.random.seed(5)
topn = 1

print(train_pop.loc[0]["lyrics"][0:400])
test_str = "Aren't you somethin' to admire? 'Cause your shine to"
generated = test_str.lower()
actual_lyric = split_str(" ;,-\\\"()[].?!:{}",  train_pop.loc[0]["lyrics"].lower().strip())
actual_lyric = " ".join([s for s in actual_lyric if s!=''][0:60])

generated = " ".join(split_str(" ;,-\\\"()[].?!:{}",  generated.strip()))

print(actual_lyric)
# generated+=sent
for i in range(20):
    x_sample = generated.split(" ")[i:i+num_words]

    x=np.zeros((1,num_words,vec_size))
    for w in range(num_words):
        x[0, w] = word_model.wv[w]
    pred = model.predict(x)
    pred = np.reshape(pred, pred.shape[1])

    ix=np.random.choice(range(len(vocab)),p=pred.ravel())
    generated+= " " + ix_word[ix]

    list_words = word_model.wv.most_similar(positive=[X[0][0]], topn=topn)
    # probs = [l[1] for l in list_words]
    # words_pred = [l[0] for l in list_words]
    # print(words_pred)
    # print(probs)

    # ix=np.random.choice(words_pred,p=probs)
    # generated+= " " + list_words[0][0]
    # print(list_words[0][0])
print(generated)

Aren't you somethin' to admire? 'Cause your shine is somethin' like a mirror And I can't help but notice You reflect in this heart of mine If you ever feel alone and The glare makes me hard to find Just know that I'm always Parallel on the other side  'Cause with your hand in my hand and a pocket full of soul I can tell you there's no place we couldn't go Just put your hand on the past I'm here tr
aren't you somethin' to admire 'cause your shine is somethin' like a mirror and i can't help but notice you reflect in this heart of mine if you ever feel alone and the glare makes me hard to find just know that i'm always parallel on the other side 'cause with your hand in my hand and a pocket full
aren't you somethin' to admire  'cause your shine to run alright and already to you this talk don't love hey friend's music about watch caught for don't you a


In [ ]:
print(generated)